# 02. Cleaning & Feature Engineering

Output: a model-ready `train_clean.csv` saved to `data/processed/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "src"))
import data_prep as dp

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

In [2]:
train = dp.load_data(RAW_DIR / "train.csv")

print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")

train: 137 rows x 43 columns


## 1. Data quality recap



In [3]:
missing = dp.missing_value_summary(train)
dupes = dp.duplicate_summary(train)

assert missing.empty, "Unexpected missing values — investigate before proceeding."
assert dupes["full_row_duplicates"] == 0, "Unexpected duplicate rows — investigate before proceeding."

print("No missing values.")
print(f"Duplicate check: {dupes}")

No missing values.
Duplicate check: {'full_row_duplicates': 0, 'content_duplicates_ignoring_id': 0}


## 2. Target: log-transform `revenue`

The 8 high-revenue outliers found in EDA are real İstanbul/İzmir restaurants, not data errors,
so they're kept. Instead of removing them, `log1p(revenue)` is added as the modeling target.

In [4]:
train["log_revenue"] = np.log1p(train["revenue"])

print(f"Skew (raw revenue):   {train['revenue'].skew():.3f}")
print(f"Skew (log_revenue):   {train['log_revenue'].skew():.3f}")

Skew (raw revenue):   2.793
Skew (log_revenue):   0.307


## 3. Restaurant age from `Open Date`

Adding `age_days` as a variable

In [5]:
train["age_days"] = dp.compute_age_days(train)

train["age_days"].describe()

count     137.000000
mean     2111.262774
std      1471.257507
min       341.000000
25%      1107.000000
50%      1773.000000
75%      2588.000000
max      6812.000000
Name: age_days, dtype: float64

## 4. Categorical encoding: `City Group`, `Type`

- `City` (34 unique values, several singletons across 137 rows) is dropped: it's too
  high-cardinality to one-hot encode safely on this sample size, and `City Group` already
  captures the Big Cities vs. Other split that mattered in EDA.
- `Type` has a singleton `DT` category. Left as-is, a train/validation split could easily put
  that one row on the wrong side and produce an unseen category. It's bucketed into `Other`
  before one-hot encoding.

In [6]:
train["type_bucketed"] = dp.bucket_rare_categories(train["Type"], min_count=5)

print("Before bucketing:")
print(train["Type"].value_counts())
print("\nAfter bucketing:")
print(train["type_bucketed"].value_counts())

Before bucketing:
Type
FC    76
IL    60
DT     1
Name: count, dtype: int64

After bucketing:
type_bucketed
FC       76
IL       60
Other     1
Name: count, dtype: int64


In [7]:
train["city_group"] = train["City Group"].rename("city_group")

cat_dummies = pd.get_dummies(
    train[["city_group", "type_bucketed"]], prefix=["city_group", "type"], drop_first=True
)

cat_dummies.head()

,city_group_Other,type_IL,type_Other
0,False,True,False
1,False,False,False
2,True,True,False
3,True,True,False
4,True,True,False


## 5. `P1`-`P37`: kept as-is

EDA found no near-constant columns (nothing to drop for lack of variance), but several highly
correlated blocks (e.g. `P10`/`P13`, `P9`/`P12`, `P24`/`P26`). Dropping or combining those is an
algorithm-specific choice — regularization handles them differently than a tree ensemble or PCA
would — so it's left to the modeling notebook rather than decided here. All 37 are carried
through unchanged.

## 6. Assemble final feature set

Combine `Id`, the engineered/encoded features, `P1`-`P37`, and both the raw and log-transformed
target into a single model-ready table. `Open Date`, `City`, `City Group`, `Type`, and
`type_bucketed` are dropped now that they've been superseded by `age_days` and the encoded
columns.

In [8]:
train_clean = pd.concat(
    [train[["Id", "age_days"]], cat_dummies, train[dp.P_COLUMNS], train[["revenue", "log_revenue"]]],
    axis=1,
)

print(f"train_clean: {train_clean.shape[0]:,} rows x {train_clean.shape[1]} columns")
train_clean.head()

train_clean: 137 rows x 44 columns


,Id,age_days,city_group_Other,type_IL,type_Other,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,P11,P12,P13,P14,P15,P16,P17,P18,P19,P20,P21,P22,P23,P24,P25,P26,P27,P28,P29,P30,P31,P32,P33,P34,P35,P36,P37,revenue,log_revenue
0,0,5647,False,True,False,4,5.0,4.0,4.0,2,2,5,4,5,5,3,5,5.0,1,2,2,2,4,5,4,1,3,3,1,1,1.0,4.0,2.0,3.0,5,3,4,5,5,4,3,4,5653753.0,15.547830
1,1,2513,False,False,False,4,5.0,4.0,4.0,1,2,5,5,5,5,1,5,5.0,0,0,0,0,0,3,2,1,3,2,0,0,0.0,0.0,3.0,3.0,0,0,0,0,0,0,0,0,6923131.0,15.750379
2,2,663,True,True,False,2,4.0,2.0,5.0,2,3,5,5,5,5,2,5,5.0,0,0,0,0,0,1,1,1,1,1,0,0,0.0,0.0,1.0,3.0,0,0,0,0,0,0,0,0,2055379.0,14.535971
3,3,1064,True,True,False,6,4.5,6.0,6.0,4,4,10,8,10,10,8,10,7.5,6,4,9,3,12,20,12,6,1,10,2,2,2.5,2.5,2.5,7.5,25,12,10,6,18,12,12,6,2675511.0,14.799651
4,4,2063,True,True,False,3,4.0,3.0,4.0,2,2,5,5,5,5,2,5,5.0,2,1,2,1,4,2,2,1,2,1,2,3,3.0,5.0,1.0,3.0,5,1,3,2,3,4,3,3,4316715.0,15.278005


In [9]:
out_path = dp.save_processed(train_clean, "train_clean.csv", PROCESSED_DIR)
print(f"Saved to {out_path}")

Saved to ../data/processed/train_clean.csv


## Summary

1. No missing values or duplicates — confirmed again on this extract.
2. `log_revenue` added as the modeling target; raw `revenue` kept
   alongside it for reference/reporting.
3. `age_days` derived from `Open Date` (days before 2015-01-01).
4. `City` dropped (34 unique values, several singletons); `City Group` and a rare-category-bucketed
   `Type` are one-hot encoded instead.
5. All 37 `P` columns kept unchanged — no near-constant ones to drop; correlated blocks are left
   for the modeling notebook to handle via regularization/PCA/feature selection.
6. Result saved to `data/processed/train_clean.csv`, ready for `03_modeling`.